 # **_[CURRENT]_** Exponential growth/accretion of information in the CDF for speakers recurring, subsequent turns.

The basic premise of this hypothesis is that information accumulates in a fractal pattern relative to any one starting point. It's not a measurement of frequency in this case, but power-law scaling in accretion patterns.

To measure accretion over time, we use the cumulative density function as defined as follows:

$$CDF(\tau) = \int_{i>1}^{\tau} I(x;y_\tau) \partial (\tau)$$

Our hypothesis/model is defined further as:

$$CDF(\tau) = e^{\beta_0}\tau^{\alpha}$$

Where $\tau$ is equal to $\Delta_t$ in other formulations, and is defined as:

$$\left[ \min t(y)_{sp(y)} > t(x) \right] - \left[t(y)_{sp(y)} > t(x) \right]_{y \in Y_{sp(y)}} + 1$$

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings;

warnings.filterwarnings('ignore')

In [ ]:
# DATA_PATH = 'updated-data/obs/lme-ready'
DATA_PATH = 'updated-data/obs'
NULL_PATH = 'updated-data/null'

In [ ]:
fs = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet') and (not f.startswith('.'))]
len(fs), fs[:3]

In [ ]:
fn = [f for f in os.listdir(NULL_PATH) if f.endswith('.parquet') and (not f.startswith('.'))]
len(fn), fn[:3]

In [ ]:
val_col = 'I'
turn_col = 'x_turn_id'
conversation_id_col = 'conversation_id'

# origin/correct
# merge_cols = [conversation_id_col, turn_col, 'self']

##### Experiments
merge_cols = [turn_col, 'y_speaker']
# merge_cols = [conversation_id_col, turn_col]

In [ ]:
# safety measure... the columns in the origina raw $I$ data.
orig_columns = [
    'index',
    'x_speaker',
    'y_speaker',
    'x_turn_id',
    'y_turn_id',
    'conversation_id',
    'corpus',
    'nx',
    'ny',
    'CoS',
    'aCoS',
    'I',
    'H'
]

## Implementation

In [ ]:
AVAR_PATH = '/'.join(DATA_PATH.split('/')[:-1])
AVAR_PATH = os.path.join(AVAR_PATH, 'INT')
if not os.path.exists(AVAR_PATH):
    os.mkdir(AVAR_PATH)

In [ ]:
# base 2 growth
m = np.array([1,2,4,16,32,64,128,200])

# base 10 growth
# m = np.array([1,10,100,1000])

In [ ]:
def integral(y,t):
    y_ = (y[:-1] + y[1:])/2
    t_ = t[1:] - t[:-1]
    return np.cumsum(y_ * t_)

### (1) Observed data

_run time 23m_

In [ ]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'obs')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fs):
    f_ = os.path.join(DATA_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    # df['EXP'] = groups[val_col]
    df['min_y_turn_id'] = groups['y_turn_id'].transform('min')
    df['mtau'] = df['y_turn_id'] - df['min_y_turn_id'] + 1

    for (xid, spy), dfi in groups:
        if len(dfi) > 1:
            spx = dfi['x_speaker'].unique()[0]
            cid = dfi['conversation_id'].unique()[0]
            nx = dfi['nx'].unique()[0]
            ymin = dfi['min_y_turn_id'].unique()[0]

            mtau = dfi['mtau'].values
            t = np.linspace(0,len(dfi)-1,len(dfi)).astype(int)

            integ = integral(dfi['I'].values, t)
            integ = integ / integ[-1]

            t = t[:-1]
            mtau = mtau[:-1]
            m_ = m[
                (m <= max(mtau))
            ]


            for mj in m_:
                n = t[mtau<=mj][-1]
                # print(xid, spy.split('-')[-1], mj, mtau[n], integ[n].item())
                # print()
                outputs += [
                    {
                        'conversation_id': cid,
                        'x_speaker': spx,
                        'y_speaker': spy,
                        'x_turn_id': xid,
                        'min_y_turn_id':ymin,
                        'nx': nx,
                        'n': n+1,
                        'mj': mj,
                        'mtau': mtau[mtau<=mj][-1],
                        'avar': integ[n]
                    }
                ]

    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs = outputs.drop_duplicates(subset=[col for col in list(outputs) if col not in ['mj', 'mtau']])
        outputs.index = range(len(outputs))
        outputs.to_parquet(
            os.path.join(output_path,f),
            compression='snappy',
            engine='fastparquet'
        )
    #############################################################

##### Last checks

In [ ]:
outputs.head(n=20)

In [ ]:
mj, mtau, integ

### (2) Null data

In [ ]:
################################################################
### Creating outputs folder
################################################################
output_path = os.path.join(AVAR_PATH, 'null')
if not os.path.exists(output_path):
    os.mkdir(output_path)



################################################################
### The show
################################################################
for f in tqdm(fn):
    f_ = os.path.join(NULL_PATH, f)
    df = pd.read_parquet(f_, engine='fastparquet')

    df = df[orig_columns]

    gc.collect()

    df = df.sort_values(by=['x_turn_id', 'y_turn_id'])
    df.index = range(len(df))

    # df['y_turn_id'] = df.groupby(by=merge_cols).cumcount() + 1
    # df['y_turn_id'] = df['x_turn_id'] + df['y_turn_id']

    # df['self'] = df['x_speaker'] == df['y_speaker']


    #############################################################
    ### Calculating AVAR
    #############################################################
    outputs = []
    groups = df.groupby(merge_cols)
    # df['EXP'] = groups[val_col]
    df['min_y_turn_id'] = groups['y_turn_id'].transform('min')
    df['mtau'] = df['y_turn_id'] - df['min_y_turn_id'] + 1

    for (xid, spy), dfi in groups:
        if len(dfi) > 1:
            spx = dfi['x_speaker'].unique()[0]
            cid = dfi['conversation_id'].unique()[0]
            nx = dfi['nx'].unique()[0]
            ymin = dfi['min_y_turn_id'].unique()[0]

            mtau = dfi['mtau'].values
            t = np.linspace(0,len(dfi)-1,len(dfi)).astype(int)

            integ = integral(dfi['I'].values, t)
            integ = integ / integ[-1]

            t = t[:-1]
            mtau = mtau[:-1]
            m_ = m[
                (m <= max(mtau))
            ]


            for mj in m_:
                n = t[mtau<=mj][-1]
                # print(xid, spy.split('-')[-1], mj, mtau[n], integ[n].item())
                # print()
                outputs += [
                    {
                        'conversation_id': cid,
                        'x_speaker': spx,
                        'y_speaker': spy,
                        'x_turn_id': xid,
                        'min_y_turn_id':ymin,
                        'nx': nx,
                        'n': n+1,
                        'mj': mj,
                        'mtau': mtau[mtau<=mj][-1],
                        'avar': integ[n]
                    }
                ]

    if len(outputs):
        outputs = pd.DataFrame(outputs)
        outputs = outputs.drop_duplicates(subset=[col for col in list(outputs) if col not in ['mj', 'mtau']])
        outputs.index = range(len(outputs))
        outputs.to_parquet(
            os.path.join(output_path,f),
            compression='snappy',
            engine='fastparquet'
        )
    #############################################################